# Smaller Model Comparison

Train 5 smaller models (3 MLPs + 2 smaller Transformers) and compare them to the 8-layer OthelloGPT (~25.3M params).

| Model | Architecture | Approx Params |
|-------|-------------|---------------|
| MLP-Small | 512 → 256 | ~1M |
| MLP-Medium | 1024 → 512 | ~3M |
| MLP-Large | 2048 → 1024 | ~5M |
| Transformer-2L | 2 layers, 512 embd, 8 heads | ~6.6M |
| Transformer-4L | 4 layers, 512 embd, 8 heads | ~12.8M |
| OthelloGPT | 8 layers, 512 embd, 8 heads | ~25.3M |

## 1. Setup & Data Loading

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import sys
import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data.dataloader import DataLoader
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt

def find_project_root():
    candidates = [
        os.getcwd(),
        os.path.dirname(os.getcwd()),
        os.path.expanduser("~/phil_proj/nothello_world"),
    ]
    for path in candidates:
        if os.path.exists(os.path.join(path, "data", "othello_synthetic")):
            return path
    raise RuntimeError("Could not find project root.")

PROJECT_ROOT = find_project_root()
os.chdir(PROJECT_ROOT)
print(f"Project root: {PROJECT_ROOT}")

from data import get_othello
from data.othello import OthelloBoardState
from mingpt.dataset import CharDataset
from mingpt.model import GPT, GPTConfig
from mingpt.utils import set_seed

set_seed(42)

# Device
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.cuda.current_device()
else:
    device = torch.device("cpu")
print(f"Using device: {device}")

In [ ]:
# Load dataset
print("Loading synthetic dataset...")
othello = get_othello(ood_num=-1, data_root=None, wthor=True)
train_dataset = CharDataset(othello)
val_dataset = CharDataset(othello.val)
print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}")
print(f"Vocab size: {train_dataset.vocab_size}, Block size: {train_dataset.block_size}")

## 2. Training Toggles

In [ ]:
# Flip to False to skip training a model (will still load from checkpoint if available)
TRAIN_MLP_SMALL = True
TRAIN_MLP_MEDIUM = True
TRAIN_MLP_LARGE = True
TRAIN_TRANSFORMER_2L = True
TRAIN_TRANSFORMER_4L = True

# Training hyperparameters
MAX_EPOCHS = 10
BATCH_SIZE = 64
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 0.1
GRAD_NORM_CLIP = 1.0

CKPT_DIR = os.path.join(PROJECT_ROOT, "ckpts")
os.makedirs(CKPT_DIR, exist_ok=True)

## 3. MLP Model Definition

In [ ]:
class OthelloMLP(nn.Module):
    """MLP baseline: token embedding -> flatten -> hidden layers -> per-position output logits."""

    def __init__(self, vocab_size, block_size, n_embd, hidden_sizes, dropout=0.1):
        super().__init__()
        self.block_size = block_size
        self.vocab_size = vocab_size
        self.tok_emb = nn.Embedding(vocab_size, n_embd)
        self.pos_emb = nn.Parameter(torch.zeros(1, block_size, n_embd))
        self.drop = nn.Dropout(dropout)

        input_dim = block_size * n_embd
        layers = []
        prev = input_dim
        for h in hidden_sizes:
            layers.extend([nn.Linear(prev, h), nn.GELU(), nn.Dropout(dropout)])
            prev = h
        # Output: for each of block_size positions, predict vocab_size logits
        layers.append(nn.Linear(prev, block_size * vocab_size))
        self.net = nn.Sequential(*layers)
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            module.weight.data.normal_(mean=0.0, std=0.02)
            if isinstance(module, nn.Linear) and module.bias is not None:
                module.bias.data.zero_()

    def get_block_size(self):
        return self.block_size

    def configure_optimizers(self, train_config):
        decay = set()
        no_decay = set()
        for mn, m in self.named_modules():
            for pn, p in m.named_parameters():
                fpn = f"{mn}.{pn}" if mn else pn
                if pn.endswith("bias"):
                    no_decay.add(fpn)
                elif pn.endswith("weight") and isinstance(m, nn.Linear):
                    decay.add(fpn)
                elif pn.endswith("weight") and isinstance(m, nn.Embedding):
                    no_decay.add(fpn)
        no_decay.add("pos_emb")
        param_dict = {pn: p for pn, p in self.named_parameters()}
        decay &= param_dict.keys()
        no_decay &= param_dict.keys()
        remaining = param_dict.keys() - (decay | no_decay)
        no_decay |= remaining  # safe default
        optim_groups = [
            {"params": [param_dict[pn] for pn in sorted(decay)], "weight_decay": train_config.weight_decay},
            {"params": [param_dict[pn] for pn in sorted(no_decay)], "weight_decay": 0.0},
        ]
        return torch.optim.AdamW(optim_groups, lr=train_config.learning_rate, betas=train_config.betas)

    def forward(self, idx, targets=None):
        b, t = idx.size()
        # Pad/truncate to block_size
        if t < self.block_size:
            pad = torch.zeros(b, self.block_size - t, dtype=idx.dtype, device=idx.device)
            idx_full = torch.cat([idx, pad], dim=1)
        else:
            idx_full = idx[:, :self.block_size]

        tok = self.tok_emb(idx_full)
        pos = self.pos_emb[:, :self.block_size, :]
        x = self.drop(tok + pos)  # [B, block_size, n_embd]
        x = x.view(b, -1)  # [B, block_size * n_embd]
        x = self.net(x)  # [B, block_size * vocab_size]
        logits = x.view(b, self.block_size, self.vocab_size)  # [B, block_size, vocab_size]

        # Trim to actual sequence length
        logits = logits[:, :t, :]

        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), targets.view(-1), ignore_index=0)
        return logits, loss

## 4. Training Helper

In [ ]:
class SimpleTrainerConfig:
    def __init__(self, **kwargs):
        self.max_epochs = 10
        self.batch_size = 64
        self.learning_rate = 3e-4
        self.betas = (0.9, 0.95)
        self.grad_norm_clip = 1.0
        self.weight_decay = 0.1
        self.num_workers = 0
        for k, v in kwargs.items():
            setattr(self, k, v)


def train_model(model, train_dataset, val_dataset, config, device, ckpt_path, model_name="model"):
    """Simple training loop (no DataParallel) compatible with MPS/CUDA/CPU."""
    model = model.to(device)
    optimizer = model.configure_optimizers(config)

    train_loader = DataLoader(
        train_dataset, shuffle=True, pin_memory=(str(device) != "mps"),
        batch_size=config.batch_size, num_workers=config.num_workers,
    )
    val_loader = DataLoader(
        val_dataset, shuffle=False, pin_memory=(str(device) != "mps"),
        batch_size=config.batch_size, num_workers=config.num_workers,
    )

    best_val_loss = float("inf")
    epoch_bar = tqdm(range(config.max_epochs), desc=f"{model_name}", unit="epoch")
    for epoch in epoch_bar:
        # Train
        model.train()
        train_losses = []
        batch_bar = tqdm(train_loader, desc=f"  Epoch {epoch+1}/{config.max_epochs} [train]", leave=False)
        for x, y in batch_bar:
            x, y = x.to(device), y.to(device)
            logits, loss = model(x, y)
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), config.grad_norm_clip)
            optimizer.step()
            train_losses.append(loss.item())
            batch_bar.set_postfix(loss=f"{loss.item():.4f}")

        # Validate
        model.eval()
        val_losses = []
        with torch.no_grad():
            for x, y in tqdm(val_loader, desc=f"  Epoch {epoch+1}/{config.max_epochs} [val]", leave=False):
                x, y = x.to(device), y.to(device)
                _, loss = model(x, y)
                val_losses.append(loss.item())
        val_loss = np.mean(val_losses)
        train_loss = np.mean(train_losses)
        epoch_bar.set_postfix(train_loss=f"{train_loss:.4f}", val_loss=f"{val_loss:.4f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), ckpt_path)

    print(f"  Best val loss: {best_val_loss:.4f} — saved to {ckpt_path}")

    # Reload best checkpoint
    model.load_state_dict(torch.load(ckpt_path, map_location=device, weights_only=False))
    return model, best_val_loss

## 5. Train / Load Each Model

In [ ]:
def count_params(model):
    return sum(p.numel() for p in model.parameters())


def make_mlp(hidden_sizes):
    return OthelloMLP(
        vocab_size=train_dataset.vocab_size,
        block_size=train_dataset.block_size,
        n_embd=128,
        hidden_sizes=hidden_sizes,
    )


def make_transformer(n_layer):
    cfg = GPTConfig(
        train_dataset.vocab_size,
        train_dataset.block_size,
        n_layer=n_layer,
        n_head=8,
        n_embd=512,
    )
    return GPT(cfg)


MODEL_SPECS = [
    ("mlp_small",        TRAIN_MLP_SMALL,        lambda: make_mlp([512, 256])),
    ("mlp_medium",       TRAIN_MLP_MEDIUM,       lambda: make_mlp([1024, 512])),
    ("mlp_large",        TRAIN_MLP_LARGE,        lambda: make_mlp([2048, 1024])),
    ("transformer_2l",   TRAIN_TRANSFORMER_2L,   lambda: make_transformer(2)),
    ("transformer_4l",   TRAIN_TRANSFORMER_4L,   lambda: make_transformer(4)),
]

trained_models = {}  # name -> model (on device)
tcfg = SimpleTrainerConfig(
    max_epochs=MAX_EPOCHS,
    batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    grad_norm_clip=GRAD_NORM_CLIP,
)

In [ ]:
for name, should_train, make_fn in MODEL_SPECS:
    ckpt_path = os.path.join(CKPT_DIR, f"{name}.ckpt")
    model = make_fn()
    n_params = count_params(model)
    print(f"\n{'='*60}")
    print(f"{name} — {n_params:,} params")

    if os.path.exists(ckpt_path):
        print(f"  Loading existing checkpoint: {ckpt_path}")
        model.load_state_dict(torch.load(ckpt_path, map_location=device, weights_only=False))
        model = model.to(device)
        model.eval()
        trained_models[name] = model
    elif should_train:
        print(f"  Training...")
        model, val_loss = train_model(model, train_dataset, val_dataset, tcfg, device, ckpt_path, model_name=name)
        model.eval()
        trained_models[name] = model
    else:
        print(f"  Skipped (toggle is False and no checkpoint found)")

## 6. Load Pretrained OthelloGPT (8-layer)

In [ ]:
gpt_ckpt = os.path.join(CKPT_DIR, "gpt_synthetic.ckpt")
assert os.path.exists(gpt_ckpt), f"OthelloGPT checkpoint not found at {gpt_ckpt}"

gpt_cfg = GPTConfig(
    train_dataset.vocab_size,
    train_dataset.block_size,
    n_layer=8, n_head=8, n_embd=512,
)
gpt_model = GPT(gpt_cfg)
gpt_model.load_state_dict(torch.load(gpt_ckpt, map_location=device, weights_only=False))
gpt_model = gpt_model.to(device)
gpt_model.eval()
trained_models["othello_gpt_8l"] = gpt_model
print(f"Loaded OthelloGPT — {count_params(gpt_model):,} params")

## 7. Evaluate All Models

In [ ]:
@torch.no_grad()
def evaluate(model, val_dataset, device, num_games=500):
    """Compute val cross-entropy loss and legal move accuracy."""
    model.eval()
    loader = DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=0)

    total_loss = 0.0
    total_tokens = 0
    total_legal = 0
    total_preds = 0

    games = val_dataset.data
    n_eval = min(num_games, len(games))

    # --- Loss over full val set (or first num_games batches) ---
    for batch_idx, (x, y) in enumerate(loader):
        if batch_idx * 64 >= n_eval:
            break
        x, y = x.to(device), y.to(device)
        logits, loss = model(x, y)
        # count non-ignored tokens
        mask = y != 0
        total_loss += loss.item() * mask.sum().item()
        total_tokens += mask.sum().item()

    avg_loss = total_loss / max(total_tokens, 1)

    # --- Legal move accuracy (per-position top-1) ---
    for game_idx in range(n_eval):
        game = games[game_idx]
        for pos in range(1, len(game)):
            context = game[:pos]
            x = torch.tensor(
                [val_dataset.stoi[s] for s in context], dtype=torch.long
            )[None, :].to(device)
            logits, _ = model(x)
            pred_idx = logits[0, -1, :].argmax().item()
            pred_move = val_dataset.itos[pred_idx]

            board = OthelloBoardState()
            board.update(context)
            valid = board.get_valid_moves()

            total_preds += 1
            if pred_move in valid:
                total_legal += 1

    legal_acc = total_legal / max(total_preds, 1)
    return avg_loss, legal_acc

In [ ]:
NUM_EVAL_GAMES = 200  # reduce for faster evaluation

results = {}  # name -> (params, val_loss, legal_acc)

for name, model in trained_models.items():
    print(f"Evaluating {name}...")
    n_params = count_params(model)
    val_loss, legal_acc = evaluate(model, val_dataset, device, num_games=NUM_EVAL_GAMES)
    results[name] = (n_params, val_loss, legal_acc)
    print(f"  {name}: params={n_params:,}, val_loss={val_loss:.4f}, legal_acc={legal_acc*100:.1f}%")

print("\nDone.")

## 8. Comparison Table & Plots

In [ ]:
# Print table
print(f"{'Model':<22} {'Params':>12} {'Val Loss':>10} {'Legal Acc':>10}")
print("-" * 56)
for name in ["mlp_small", "mlp_medium", "mlp_large", "transformer_2l", "transformer_4l", "othello_gpt_8l"]:
    if name in results:
        params, vl, la = results[name]
        print(f"{name:<22} {params:>12,} {vl:>10.4f} {la*100:>9.1f}%")

In [ ]:
# Bar charts
display_order = ["mlp_small", "mlp_medium", "mlp_large", "transformer_2l", "transformer_4l", "othello_gpt_8l"]
names = [n for n in display_order if n in results]
params_list = [results[n][0] for n in names]
losses = [results[n][1] for n in names]
accs = [results[n][2] * 100 for n in names]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

colors = ["#4c72b0" if "mlp" in n else "#dd8452" for n in names]

axes[0].barh(names, [p / 1e6 for p in params_list], color=colors)
axes[0].set_xlabel("Parameters (M)")
axes[0].set_title("Model Size")

axes[1].barh(names, losses, color=colors)
axes[1].set_xlabel("Val Cross-Entropy Loss")
axes[1].set_title("Validation Loss")

axes[2].barh(names, accs, color=colors)
axes[2].set_xlabel("Legal Move Accuracy (%)")
axes[2].set_title("Legal Move Accuracy")

plt.tight_layout()
plt.show()

In [ ]:
# Scatter: params vs accuracy
fig, ax = plt.subplots(figsize=(8, 5))
for n in names:
    p, vl, la = results[n]
    marker = "s" if "mlp" in n else "o"
    color = "#4c72b0" if "mlp" in n else "#dd8452"
    ax.scatter(p / 1e6, la * 100, s=100, marker=marker, color=color, zorder=5)
    ax.annotate(n, (p / 1e6, la * 100), textcoords="offset points", xytext=(5, 5), fontsize=9)

ax.set_xlabel("Parameters (M)")
ax.set_ylabel("Legal Move Accuracy (%)")
ax.set_title("Parameters vs Legal Move Accuracy")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()